# 캡스톤 템플릿 노트북 — 사전 등록 → 출처 → 분할·누출 → baseline → 평가 → 민감도 → 대조 → 보고

**등급**: 🟢 Mac 로컬 · **예상 시간**: 약 5분 (데모 설정) · **다운로드**: 없음 (모듈 02의 ProteinGym 캐시와 ESM-2 8M 캐시)

이 노트북은 **프로젝트 골격**이다. 데모는 트랙 A(단백질 변이 효과)의 가장 작은 예 — "동결 ESM-2 8M 임베딩 probe가 위치 정보를 쓰는 단순 baseline보다, *새 위치*를 예측할 때 나은가?" — 로 채워져 있다. 자기 프로젝트에서는 **`✏️`로 표시한 셀만** 바꾸고, 나머지 구조(출처 기록, 누출 측정, baseline, 짝지은 비교, 그룹 부트스트랩, 민감도, 음성 대조, 보고)는 유지한다.

**이 골격이 강제하는 것** (모듈 01~06에서 실제로 틀렸거나 틀리기 쉬웠던 것):
1. 설정(분할·지표·결정 규칙)을 첫 셀에서 **결과를 보기 전에** 고정한다.
2. 데이터 파일 해시, 모델 리비전, 패키지 버전을 기록한다 (모듈 05: 고정하지 않은 리비전이 섞인 적이 있다).
3. 분할마다 **train과의 겹침을 숫자로 잰다** — 분할 이름을 믿지 않는다 (모듈 05).
4. baseline은 학습 없는 것/단순한 것/무작위 초기화를 포함하고 **같은 절차**로 평가한다 (모듈 04·05).
5. 비교는 **같은 분할에서 짝지어**, 구간은 상관된 단위(위치 블록)의 **그룹 부트스트랩**으로 (모듈 04·05).
6. 한 번에 한 요인만 바꾸는 **민감도**와, 파이프라인이 우연 수준을 내는지 보는 **음성 대조**를 넣는다 (모듈 05·06).
7. 설정은 test로 고르지 않는다 — 여기서는 사전에 고정했고 민감도는 보고만 한다 (모듈 04·06).

## 0. ✏️ 사전 등록 설정 (결과를 보기 전에 정한다)

In [ ]:
RUN_NAME = "demo_dn7a_probe_vs_baselines"

# ✏️ 데이터·모델
ASSAY = "DN7A_SACS2_Tsuboyama_2023_1JIC"                       # ProteinGym 단일 치환 assay (모듈 02 캐시)
MODEL = ("facebook/esm2_t6_8M_UR50D", "c731040fcd8d73dceaa04b0a8e6329b345b0f5df")      # (저장소, 리비전)

# ✏️ 분할·반복·불확실성
SPLITS = ["random", "position-block"]
BLOCK = 10                                                       # position-block 크기 (연속 위치 수)
SEEDS = [0, 1, 2, 3, 4]                                          # 시드마다 분할과 모델 초기화가 모두 바뀐다
BOOT = 500                                                       # 그룹 부트스트랩 재표본 수

# ✏️ 사전 지정한 주 비교와 결정 규칙
PRIMARY = dict(split="position-block", method="probe", baseline="B2")
DECISION_RULE = "주 비교의 Δ(= 지표_method − 지표_baseline; 시드 평균, 위치 블록 그룹 부트스트랩 95% 구간)가 0을 포함하지 않고 양수이면 '지지', 그 밖에는 '지지되지 않음'"
SENSITIVITY_BLOCKS = [3, 5, 10]                                  # 한 번에 한 요인: 블록 크기 (길이 55 단백질에서 블록이 5개 이상 남는 크기만 가능: 3→19개, 5→11개, 10→6개)

print("RUN:", RUN_NAME)
print("주 비교:", PRIMARY)
print("결정 규칙:", DECISION_RULE)

In [ ]:
import hashlib
import importlib.metadata as md
import json
import os
import subprocess
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.Align import substitution_matrices
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, ShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
CACHE = (ROOT / ".cache") if ROOT is not None else Path("cache")
os.environ.setdefault("HF_HOME", str(CACHE / "hf"))
PG = CACHE / "proteingym"
OUT = CACHE / "capstone_runs" / RUN_NAME                         # git 제외 폴더: 실행 기록은 여기에만 쓴다
OUT.mkdir(parents=True, exist_ok=True)

import torch
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer, EsmModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE, "| 출력 폴더:", OUT)

## 1. 출처 기록 (provenance)
해시·리비전·버전·커밋을 `provenance.json`에 남긴다. 결과 표를 다시 만들 때 "무엇으로 만들었는가"에 답할 수 있어야 한다. 라이선스는 **확인한 근거를 적고, 확인하지 못했으면 '미확인'**이라고 적는다(모듈 06 §7).

In [ ]:
def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()


def git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=ROOT).stdout.strip() or "(git 정보 없음)"
    except Exception:
        return "(git 정보 없음)"


ZIP, REF = PG / "DMS_ProteinGym_substitutions.zip", PG / "DMS_substitutions.csv"
assert ZIP.exists() and REF.exists(), "모듈 02 노트북 01로 ProteinGym 파일을 먼저 받을 것"
PROV = dict(
    run=RUN_NAME, time=time.strftime("%Y-%m-%d %H:%M:%S"), repo_commit=git_commit(),
    data={"ProteinGym zip sha256": sha256(ZIP)[:16] + "…", "reference csv sha256": sha256(REF)[:16] + "…", "assay": ASSAY},
    model={"repo": MODEL[0], "revision": MODEL[1]},
    packages={k: md.version(k) for k in ["numpy", "pandas", "scikit-learn", "scipy", "torch", "transformers"]},
    seeds=SEEDS, splits=SPLITS, block=BLOCK, boot=BOOT, primary=PRIMARY, decision_rule=DECISION_RULE,
    licenses={"ESM-2 weights": "MIT (Hugging Face 메타데이터)", "ProteinGym 저장소 코드": "MIT (GitHub API)", "ProteinGym 개별 DMS 데이터": "미확인 (assay별 원 출처 확인 필요)"},
)
(OUT / "provenance.json").write_text(json.dumps(PROV, ensure_ascii=False, indent=1))
print(json.dumps(PROV, ensure_ascii=False, indent=1)[:1400])

## 2. 데이터 점검
불러온 직후에 **자체 점검(assert)**을 둔다: wt 잔기 일치, 서열 길이, 중복. 점검이 깨지면 이후 결과는 의미가 없다.

In [ ]:
ref = pd.read_csv(REF)


def load_assay(dms_id):
    row = ref[ref.DMS_id == dms_id].iloc[0]
    with zipfile.ZipFile(ZIP) as z:
        df = pd.read_csv(z.open(f"DMS_ProteinGym_substitutions/{row.DMS_filename}"))
    df = df[~df.mutant.str.contains(":")].copy()                                      # 단일 치환만
    df["wt"], df["pos"], df["mt"] = df.mutant.str[0], df.mutant.str[1:-1].astype(int), df.mutant.str[-1]
    assert all(row.target_seq[p - 1] == w for p, w in zip(df.pos, df.wt)), "wt 잔기가 target_seq와 다름"
    assert df.mutant.is_unique, "같은 변이가 중복되어 있다"
    assert df.mutated_sequence.str.len().nunique() == 1, "길이가 같지 않은 서열이 있다"
    return row.target_seq, df.reset_index(drop=True), row


WT, DF, ROW = load_assay(ASSAY)
Y = DF.DMS_score.values.astype(float)
print(f"{ASSAY}: 길이 {len(WT)} | 단일 치환 {len(DF)}개 | 위치 {DF.pos.nunique()}개 | 유형 {ROW.coarse_selection_type}/{ROW.selection_type}")
print(f"점수 분포: 중앙값 {np.median(Y):.2f}, 5/95% [{np.percentile(Y, 5):.2f}, {np.percentile(Y, 95):.2f}] | 결측 {int(np.isnan(Y).sum())}")

## 3. 분할과 누출 측정
분할이 막으려는 것을 먼저 적는다: **random**은 아무것도 막지 않고, **position-block**은 "같은 위치의 다른 치환이 train에 있는" 경로를 막는다. 이름을 믿지 말고 숫자로 확인한다 — test 변이 중 (a) 위치가 train에도 있는 비율, (b) 같은 (wt→mt) 쌍이 train에 있는 비율, (c) train의 가장 가까운 위치까지의 거리 중앙값.

In [ ]:
def make_split(kind, seed, block=BLOCK):
    if kind == "random":
        tr, te = next(ShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(DF))
        groups = np.arange(len(DF))
    else:
        groups = (DF.pos.values - 1) // block
        assert len(np.unique(groups)) >= 5, f"블록 크기 {block}이 너무 커서 그룹이 {len(np.unique(groups))}개뿐이다 — 분할·내부 CV가 성립하지 않는다"
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed).split(DF, groups=groups))
    return tr, te, groups


def leak_stats(tr, te):
    pos_tr = np.sort(DF.pos.values[tr])
    seen_pos = np.isin(DF.pos.values[te], pos_tr).mean()
    pair_tr = set(zip(DF.wt.values[tr], DF.mt.values[tr]))
    seen_pair = np.mean([(w, m) in pair_tr for w, m in zip(DF.wt.values[te], DF.mt.values[te])])
    dist = np.min(np.abs(DF.pos.values[te][:, None] - pos_tr[None, :]), axis=1)
    return dict(test_위치가_train에_있음=seen_pos, 같은_wt_mt쌍이_train에_있음=seen_pair, train까지_위치거리_중앙값=float(np.median(dist)), n_test=len(te))


rows = []
for kind in SPLITS:
    st = pd.DataFrame([leak_stats(*make_split(kind, s)[:2]) for s in SEEDS])
    rows.append(dict(분할=kind, **st.mean().round(3).to_dict()))
print(pd.DataFrame(rows).to_string(index=False))

## 4. 특징: baseline과 모델
- **B1**: wt·mutant 원-핫 + BLOSUM62 (위치 정보 없음), **B2**: B1 + 위치 원-핫 (같은 위치가 train에 있으면 유리).
- **probe**: 동결 ESM-2 8M, 변이 위치의 마지막 층 은닉 상태. **probe-randinit**: 같은 구조의 *무작위 초기화* 모델(시드 0) — 사전학습의 기여를 분리하는 대조군.
- 모든 방법은 **같은 Ridge 파이프라인**(표준화 → Ridge, 규제 세기는 학습 안 그룹 CV로 Spearman 기준 선택)을 쓴다.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
BLOSUM = substitution_matrices.load("BLOSUM62")


def onehot(chars):
    M = np.zeros((len(chars), len(AA)))
    M[np.arange(len(chars)), [AA.index(c) for c in chars]] = 1
    return M


blos = np.array([BLOSUM[w][m] for w, m in zip(DF.wt, DF.mt)], dtype=float)[:, None]
B1 = np.hstack([onehot(DF.wt), onehot(DF.mt), blos])
P = np.zeros((len(DF), len(WT)))
P[np.arange(len(DF)), DF.pos.values - 1] = 1
B2 = np.hstack([B1, P])

mdir = Path(snapshot_download(MODEL[0], revision=MODEL[1], allow_patterns=["model.safetensors", "*.json", "vocab.txt"]))
tok = AutoTokenizer.from_pretrained(mdir)
ids_all = tok(DF.mutated_sequence.tolist(), return_tensors="pt")["input_ids"]
pos_all = torch.tensor(DF.pos.values)


@torch.no_grad()
def site_features(model, batch=128):
    model = model.eval().to(DEVICE)
    out = []
    for s in range(0, len(ids_all), batch):
        ids = ids_all[s:s + batch].to(DEVICE)
        h = model(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state
        out.append(h[torch.arange(len(ids)), pos_all[s:s + batch].to(DEVICE)].float().cpu())
    return torch.cat(out).numpy()


t0 = time.time()
X_probe = site_features(EsmModel.from_pretrained(mdir, add_pooling_layer=False))
torch.manual_seed(0)
X_rand = site_features(EsmModel(AutoConfig.from_pretrained(mdir), add_pooling_layer=False))
print(f"임베딩 {time.time() - t0:.0f}초 | probe {X_probe.shape} | 무작위 초기화 {X_rand.shape}")
FEATS = {"B1": B1, "B2": B2, "probe": X_probe, "probe-randinit": X_rand}

## 5. 평가
시드마다 (분할 → 모든 방법)을 **같은 train/test**로 평가한다. 예측을 저장해 두고 짝지은 차이와 부트스트랩에 재사용한다. 음성 대조 **probe-shuffled**: 학습 라벨을 섞어 학습 — **시드 평균이 0 근처**이고 부호가 일정하지 않아야 파이프라인에 누출이 없다. 단, test가 소수의 위치에 몰려 있으면(유효 표본 수 ≈ test 위치 수) 시드 하나의 우연 변동은 ±0.3까지도 커진다 — 한 시드의 |ρ|가 0.3이어도 그것만으로 누출 신호는 아니다.

In [ ]:
def rho(y, p):
    r = spearmanr(y, p).statistic
    return -1.0 if np.isnan(r) else float(r)


def fit_predict(X, y, groups, tr, te):
    pipe = make_pipeline(StandardScaler(), Ridge())
    n_cv = min(3, len(np.unique(groups[tr])))                                      # 학습 안 그룹 수가 3보다 작으면 접힘 수를 줄인다
    gs = GridSearchCV(pipe, {"ridge__alpha": np.logspace(-1, 5, 13)}, scoring=make_scorer(rho), cv=GroupKFold(n_splits=n_cv).split(X[tr], y[tr], groups[tr]))
    gs.fit(X[tr], y[tr])
    return gs.predict(X[te])


PRED = {}                                                       # (분할, 시드, 방법) -> (te, pred)
rows = []
t0 = time.time()
for kind in SPLITS:
    for seed in SEEDS:
        tr, te, groups = make_split(kind, seed)
        for name, X in FEATS.items():
            p = fit_predict(X, Y, groups, tr, te)
            PRED[(kind, seed, name)] = (te, p)
            rows.append(dict(split=kind, seed=seed, method=name, rho=rho(Y[te], p)))
        y_shuf = Y.copy()
        y_shuf[tr] = np.random.default_rng(seed).permutation(Y[tr])
        p = fit_predict(X_probe, y_shuf, groups, tr, te)
        PRED[(kind, seed, "probe-shuffled")] = (te, p)
        rows.append(dict(split=kind, seed=seed, method="probe-shuffled", rho=rho(Y[te], p)))
RES = pd.DataFrame(rows)
print(f"평가 {time.time() - t0:.0f}초")
tab = RES.groupby(["split", "method"]).rho.agg(["mean", "std"]).round(3).unstack(0)
print(tab.to_string())

### 주 비교: 짝지은 차이와 위치 블록 그룹 부트스트랩
사전 지정한 주 비교(`PRIMARY`)의 Δ = probe − B2를 시드별로 구하고(같은 train/test), 구간은 **각 시드의 test에서 위치 블록을 재표본**한 뒤 시드에 걸쳐 평균해 만든다(조건부 구간: 이미 계산한 예측에 대한 구간이며 모델 재학습·분할 변동은 담지 않는다).

In [ ]:
def boot_delta(kind, m1, m2, block=BLOCK, B=BOOT, seed=123):
    rng = np.random.default_rng(seed)
    per_seed = []
    for s in SEEDS:
        te, p1 = PRED[(kind, s, m1)]
        _, p2 = PRED[(kind, s, m2)]
        per_seed.append((te, p1, p2, (DF.pos.values[te] - 1) // block))
    point = np.mean([rho(Y[te], p1) - rho(Y[te], p2) for te, p1, p2, _ in per_seed])
    draws = np.zeros(B)
    for b in range(B):
        ds = []
        for te, p1, p2, g in per_seed:
            u = np.unique(g)
            pick = rng.choice(u, size=len(u), replace=True)
            idx = np.concatenate([np.where(g == k)[0] for k in pick])
            ds.append(rho(Y[te][idx], p1[idx]) - rho(Y[te][idx], p2[idx]))
        draws[b] = np.mean(ds)
    wins = sum(rho(Y[te], p1) > rho(Y[te], p2) for te, p1, p2, _ in per_seed)
    return point, np.percentile(draws, [2.5, 97.5]), wins


rows = []
for kind in SPLITS:
    for m1, m2 in [("probe", "B2"), ("probe", "B1"), ("probe", "probe-randinit")]:
        pt, (lo, hi), wins = boot_delta(kind, m1, m2)
        rows.append({"분할": kind, "비교": f"{m1} − {m2}", "Δ": f"{pt:+.3f} [{lo:+.3f}, {hi:+.3f}]", "이긴 시드": f"{wins}/{len(SEEDS)}"})
CMP = pd.DataFrame(rows)
print(CMP.to_string(index=False))

pt, (lo, hi), wins = boot_delta(PRIMARY["split"], PRIMARY["method"], PRIMARY["baseline"])
verdict = "지지" if (lo > 0) else "지지되지 않음"
print(f"\n[사전 지정 결정] {PRIMARY} → Δ = {pt:+.3f} [{lo:+.3f}, {hi:+.3f}], 이긴 시드 {wins}/{len(SEEDS)} → '{verdict}'")

## 6. 민감도: 한 번에 한 요인 (블록 크기)
주 비교가 블록 크기에 좌우되는가? **설정을 이 표로 고르지 않는다** — 결론이 사전 지정한 설정에 얼마나 의존하는지 보고하는 용도다. 블록 크기를 바꾸면 분할이 달라지므로 baseline도 같은 분할에서 다시 평가한다.

In [ ]:
rows = []
t0 = time.time()
for blk in SENSITIVITY_BLOCKS:
    store = {}
    for seed in SEEDS:
        tr, te, groups = make_split("position-block", seed, block=blk)
        for name in ["B2", "probe"]:
            store[(seed, name)] = (te, fit_predict(FEATS[name], Y, groups, tr, te))
    d = [rho(Y[store[(s, 'probe')][0]], store[(s, 'probe')][1]) - rho(Y[store[(s, 'B2')][0]], store[(s, 'B2')][1]) for s in SEEDS]
    rows.append({"블록 크기": blk, "Δ(probe−B2) 시드 평균": np.mean(d), "시드별": ", ".join(f"{x:+.2f}" for x in d), "이긴 시드": f"{int(np.sum(np.array(d) > 0))}/{len(SEEDS)}"})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"민감도 {time.time() - t0:.0f}초")

## 7. 결과 저장과 보고 문장 점검
표를 파일로 저장하고, 보고서에 붙일 **숫자 문장**을 코드에서 만든다 — 서술의 숫자를 손으로 옮기다 틀리는 일(모듈 04·05)을 줄이기 위해서다.

In [ ]:
(OUT / "results_by_method.csv").write_text(RES.to_csv(index=False))
(OUT / "comparisons.txt").write_text(CMP.to_string(index=False))
print("저장:", sorted(p.name for p in OUT.iterdir()))
m = RES.groupby(["split", "method"]).rho.mean()
print("\n[자동 숫자 문장]")
for kind in SPLITS:
    print(f"- {kind}: " + ", ".join(f"{k} {m[(kind, k)]:.2f}" for k in ["B1", "B2", "probe", "probe-randinit", "probe-shuffled"]))
sh = RES[RES.method == "probe-shuffled"].groupby("split").rho.agg(["mean", "std", lambda v: v.abs().max()])
for kind in SPLITS:
    print(f"- 음성 대조(라벨 섞음) {kind}: 시드 평균 {sh.loc[kind].iloc[0]:+.2f}, SD {sh.loc[kind].iloc[1]:.2f}, |ρ| 최대 {sh.loc[kind].iloc[2]:.2f} (평균이 0 근처인지, 효과 크기보다 충분히 작은지 본다; 위치 블록 분할은 유효 표본 수가 작아 변동이 크다)")

## 8. ✏️ 한계와 미실시 목록 (직접 채운다)
보고서에는 **하지 않은 것**을 적는다. 이 데모에서 예시:
- 한 assay, 한 모델 크기(8M), 시드 5개, 규제 외 하이퍼파라미터 없음(Ridge만).
- 구간은 고정된 예측에 대한 조건부 구간이다. 분할·재학습 변동은 포함하지 않는다.
- 데이터 라이선스 중 assay별 원 출처는 확인하지 못했다.
- 결과를 다른 assay·모델로 일반화하지 않는다.